# 🌌 Deepfake Detection AI — Free Google Colab T4 GPU Hosting

Run the complete **Multi-Modal Deepfake Forensic Platform** on a **Free NVIDIA Tesla T4 GPU (15 GB VRAM)** with a public HTTPS link accessible worldwide.

### ⚡ Setup Instructions (Takes 1 minute):
1. In the top menu, go to **Runtime** > **Change runtime type**.
2. Select **T4 GPU** under Hardware accelerator, then click **Save**.
3. Run the cells below in order.

### 1. Verify NVIDIA GPU & CUDA Acceleration

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU:     {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not detected! Please go to Runtime -> Change runtime type -> T4 GPU")

### 2. Clone Project Repository (with Model Weights via Git LFS)

In [ ]:
import os

# Clean up any previous runs
!rm -rf /content/deepfake_app

# Clone the repository directly from Hugging Face Spaces (includes best_model.pth via LFS)
!git clone https://huggingface.co/spaces/Chaman-sengar07/deepfake-detection-ai /content/deepfake_app

%cd /content/deepfake_app

# Verify files
!ls -la checkpoints

### 3. Install System & Python Dependencies

In [ ]:
# Install audio/video system packages
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1

# Install Python packages (FastAPI, MTCNN, Grad-CAM, Transformers, Librosa, etc.)
!pip install -q fastapi uvicorn python-multipart facenet-pytorch grad-cam opencv-python-headless transformers librosa soundfile gradio

### 4. Download Cloudflare Tunnel (Free Public HTTPS Tunnel)

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
print("✅ Cloudflare Tunnel binary ready.")

### 5. Launch FastAPI Backend & Expose Public URL 🚀
Run this cell to start your application with GPU acceleration and generate a public HTTPS URL.

In [ ]:
import subprocess
import time
import re
import sys

# Ensure runs directory exists
os.makedirs("/content/deepfake_app/runs", exist_ok=True)

# Terminate any running server on port 8000
!fuser -k 8000/tcp || true

print("[*] Starting Deepfake Detection Engine on GPU...")
# Start FastAPI server via Uvicorn in the background
server_cmd = [sys.executable, "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"]
server_process = subprocess.Popen(server_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Wait 5 seconds for model and CUDA initialization
time.sleep(5)

print("[*] Establishing secure Cloudflare tunnel...")
tunnel_cmd = ["/content/deepfake_app/cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000"]
tunnel_process = subprocess.Popen(tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

public_url = None
# Scan tunnel logs for public URL
for _ in range(60):
    line = tunnel_process.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(0.5)

if public_url:
    print("\n" + "="*65)
    print("🎉 DEEPFAKE DETECTION AI IS LIVE IN PRODUCTION!")
    print(f"👉 Open Public URL:  {public_url}")
    print(f"👉 API Documentation: {public_url}/docs")
    print("="*65 + "\n")
    print("Keep this cell running to keep the app online.")
else:
    print("⚠️ Could not automatically detect Cloudflare URL. Check logs below:")

# Stream server logs
try:
    while True:
        out = server_process.stdout.readline()
        if out:
            print(out.strip())
        time.sleep(0.1)
except KeyboardInterrupt:
    print("Stopping server...")
    server_process.terminate()
    tunnel_process.terminate()
